# 🔬 Additional A100 Experiments
**SDXL + Extra resolutions + Batched inference + Mistral-7B**

Run ALL cells in order. Results save to Google Drive.
Expected runtime: ~60-90 min on A100.

In [ ]:
# Cell 1: Install & Setup
!pip install -q transformers==4.47.1 diffusers accelerate pynvml audiocraft scipy
import torch, time, json, os, gc, datetime
import numpy as np
import pynvml
from threading import Thread, Event
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/project_energy_results'
os.makedirs(SAVE_DIR, exist_ok=True)
pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)
GPU_NAME = pynvml.nvmlDeviceGetName(handle)
if isinstance(GPU_NAME, bytes): GPU_NAME = GPU_NAME.decode()
gpu_tag = GPU_NAME.replace(' ','-')
REPEATS = 30
print(f'GPU: {GPU_NAME}, PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}')

class PowerMonitor:
    def __init__(self):
        self.samples = []
        self._stop = Event()
    def _record(self):
        while not self._stop.is_set():
            try:
                p = pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0
                self.samples.append((time.time(), p))
            except: pass
            time.sleep(0.1)
    def start(self):
        self.samples = []
        self._stop.clear()
        self._thread = Thread(target=self._record, daemon=True)
        self._thread.start()
    def stop(self):
        self._stop.set()
        self._thread.join()
        if len(self.samples) < 2: return {}
        times = [s[0] for s in self.samples]
        powers = [s[1] for s in self.samples]
        dt = [times[i+1]-times[i] for i in range(len(times)-1)]
        energy = sum((powers[i]+powers[i+1])/2*dt[i] for i in range(len(dt)))
        return {'total_energy_j': round(energy,2), 'avg_power_w': round(np.mean(powers),2),
                'max_power_w': round(max(powers),2), 'duration_sec': round(times[-1]-times[0],2),
                'samples': len(self.samples)}
print('Setup complete')

In [ ]:
# Cell 2: SD v1.5 extra resolutions (128², 640²)
from diffusers import StableDiffusionPipeline
pipe = StableDiffusionPipeline.from_pretrained('stable-diffusion-v1-5/stable-diffusion-v1-5',
    torch_dtype=torch.float16).to('cuda')
prompt = 'A serene landscape with mountains and a lake at sunset, highly detailed, 8k'
results = []
for res in [128, 640]:
    print(f'  SD v1.5 {res}x{res}, 20 steps')
    _ = pipe(prompt, height=res, width=res, num_inference_steps=20,
            guidance_scale=7.5, generator=torch.Generator('cuda').manual_seed(0))
    for run in range(1, REPEATS+1):
        torch.cuda.synchronize()
        pm = PowerMonitor(); pm.start()
        t0 = time.time()
        _ = pipe(prompt, height=res, width=res, num_inference_steps=20,
                guidance_scale=7.5, generator=torch.Generator('cuda').manual_seed(run))
        torch.cuda.synchronize()
        stats = pm.stop()
        stats.update({'generation_time_sec': round(time.time()-t0,2), 'resolution': f'{res}x{res}',
                      'steps': 20, 'modality': 'image', 'model': 'SD-v1-5', 'params_B': 0.9,
                      'run': run, 'hardware': GPU_NAME, 'backend': 'cuda'})
        results.append(stats)
with open(f'{SAVE_DIR}/exp2_image_extra_{gpu_tag}.json','w') as f: json.dump(results,f,indent=2)
del pipe; gc.collect(); torch.cuda.empty_cache()
print(f'✅ SD v1.5 extra done: {len(results)} runs')

In [ ]:
# Cell 3: AnimateDiff extra frames (6, 16)
from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler
adapter = MotionAdapter.from_pretrained('guoyww/animatediff-motion-adapter-v1-5-3',
    torch_dtype=torch.float16)
vpipe = AnimateDiffPipeline.from_pretrained('stable-diffusion-v1-5/stable-diffusion-v1-5',
    motion_adapter=adapter, torch_dtype=torch.float16).to('cuda')
vpipe.scheduler = DDIMScheduler.from_config(vpipe.scheduler.config,
    beta_schedule='linear', clip_sample=False, timestep_spacing='linspace', steps_offset=1)
prompt = 'A serene landscape with mountains and a lake at sunset, highly detailed, 8k'
results = []
for frames in [6, 16]:
    print(f'  AnimateDiff {frames}f, 20 steps, 256x256')
    _ = vpipe(prompt, num_frames=frames, height=256, width=256,
             num_inference_steps=20, generator=torch.Generator('cuda').manual_seed(0))
    for run in range(1, REPEATS+1):
        torch.cuda.synchronize()
        pm = PowerMonitor(); pm.start()
        t0 = time.time()
        _ = vpipe(prompt, num_frames=frames, height=256, width=256,
                 num_inference_steps=20, generator=torch.Generator('cuda').manual_seed(run))
        torch.cuda.synchronize()
        stats = pm.stop()
        stats.update({'generation_time_sec': round(time.time()-t0,2), 'frames': frames,
                      'steps': 20, 'resolution': '256x256', 'modality': 'video',
                      'model': 'AnimateDiff-v1.5', 'run': run, 'hardware': GPU_NAME, 'backend': 'cuda'})
        results.append(stats)
with open(f'{SAVE_DIR}/exp3_video_extra_{gpu_tag}.json','w') as f: json.dump(results,f,indent=2)
del vpipe, adapter; gc.collect(); torch.cuda.empty_cache()
print(f'✅ AnimateDiff extra done: {len(results)} runs')

In [ ]:
# Cell 4: SDXL (second image model)
from diffusers import StableDiffusionXLPipeline
pipe = StableDiffusionXLPipeline.from_pretrained('stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16, variant='fp16', use_safetensors=True).to('cuda')
prompt = 'A serene landscape with mountains and a lake at sunset, highly detailed, 8k'
results = []
for res in [512, 768, 1024]:
    print(f'  SDXL {res}x{res}, 20 steps')
    _ = pipe(prompt, height=res, width=res, num_inference_steps=20,
            guidance_scale=7.5, generator=torch.Generator('cuda').manual_seed(0))
    for run in range(1, REPEATS+1):
        torch.cuda.synchronize()
        pm = PowerMonitor(); pm.start()
        t0 = time.time()
        _ = pipe(prompt, height=res, width=res, num_inference_steps=20,
                guidance_scale=7.5, generator=torch.Generator('cuda').manual_seed(run))
        torch.cuda.synchronize()
        stats = pm.stop()
        stats.update({'generation_time_sec': round(time.time()-t0,2), 'resolution': f'{res}x{res}',
                      'steps': 20, 'modality': 'image', 'model': 'SDXL-base-1.0', 'params_B': 3.5,
                      'run': run, 'hardware': GPU_NAME, 'backend': 'cuda'})
        results.append(stats)
with open(f'{SAVE_DIR}/exp5_sdxl_{gpu_tag}.json','w') as f: json.dump(results,f,indent=2)
del pipe; gc.collect(); torch.cuda.empty_cache()
print(f'✅ SDXL done: {len(results)} runs')

In [ ]:
# Cell 5: Mistral-7B (second text model)
from transformers import AutoModelForCausalLM, AutoTokenizer
model_id = 'mistralai/Mistral-7B-Instruct-v0.3'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map='cuda')
prompt = 'Write a short story about a robot learning to paint.'
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
results = []
for max_tok in [64, 128, 256]:
    print(f'  Mistral-7B {max_tok} tokens')
    _ = model.generate(**inputs, max_new_tokens=max_tok, do_sample=False)
    for run in range(1, REPEATS+1):
        torch.cuda.synchronize()
        pm = PowerMonitor(); pm.start()
        t0 = time.time()
        out = model.generate(**inputs, max_new_tokens=max_tok, do_sample=False)
        torch.cuda.synchronize()
        stats = pm.stop()
        stats.update({'generation_time_sec': round(time.time()-t0,2), 'max_tokens': max_tok,
                      'modality': 'text', 'model': 'Mistral-7B-Instruct', 'params_B': 7.2,
                      'run': run, 'hardware': GPU_NAME, 'backend': 'cuda'})
        results.append(stats)
with open(f'{SAVE_DIR}/exp6_mistral_{gpu_tag}.json','w') as f: json.dump(results,f,indent=2)
del model, tokenizer; gc.collect(); torch.cuda.empty_cache()
print(f'✅ Mistral-7B done: {len(results)} runs')

In [ ]:
# Cell 6: Batched inference — Phi-3 (1, 2, 4, 8 batch)
from transformers import AutoModelForCausalLM, AutoTokenizer
model_id = 'microsoft/Phi-3-mini-4k-instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16,
    trust_remote_code=True, device_map='cuda')
prompt = 'Write a short story about a robot learning to paint.'
results = []
for batch_size in [1, 2, 4, 8]:
    inputs = tokenizer([prompt]*batch_size, return_tensors='pt', padding=True).to('cuda')
    max_tok = 256
    print(f'  Phi-3 batch={batch_size}, 256 tokens')
    _ = model.generate(**inputs, max_new_tokens=max_tok, do_sample=False)
    for run in range(1, REPEATS+1):
        torch.cuda.synchronize()
        pm = PowerMonitor(); pm.start()
        t0 = time.time()
        out = model.generate(**inputs, max_new_tokens=max_tok, do_sample=False)
        torch.cuda.synchronize()
        stats = pm.stop()
        per_query_energy = round(stats.get('total_energy_j',0) / batch_size, 2)
        per_query_time = round((time.time()-t0) / batch_size, 2)
        stats.update({'generation_time_sec': round(time.time()-t0,2), 'max_tokens': max_tok,
                      'batch_size': batch_size, 'per_query_energy_j': per_query_energy,
                      'per_query_time_sec': per_query_time,
                      'modality': 'text_batched', 'model': 'Phi-3-mini-4k', 'params_B': 3.8,
                      'run': run, 'hardware': GPU_NAME, 'backend': 'cuda'})
        results.append(stats)
with open(f'{SAVE_DIR}/exp7_batched_{gpu_tag}.json','w') as f: json.dump(results,f,indent=2)
del model, tokenizer; gc.collect(); torch.cuda.empty_cache()
print(f'✅ Batched inference done: {len(results)} runs')

In [ ]:
# Cell 7: Summary
print('\n🎉 ALL ADDITIONAL EXPERIMENTS COMPLETE!')
print(f'Timestamp: {datetime.datetime.now()}')
print(f'GPU: {GPU_NAME}')
import glob
for f in sorted(glob.glob(f'{SAVE_DIR}/exp*_{gpu_tag}.json')):
    with open(f) as fh:
        data = json.load(fh)
    print(f'  {os.path.basename(f)}: {len(data)} records')